# Week 22, Day 2: The phase plane and the trace-determinant plane

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w022/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A classifier for 2 × 2 systems

Write `classify(A)` using only the trace T and determinant D. Return 'saddle', 'stable node', 'unstable node', 'stable spiral', 'unstable spiral' or 'centre' (treat |T| below 1e-12 with complex eigenvalues as a centre).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def classify(A):
    T, D = np.trace(A), np.linalg.det(A)
    if D < 0:
        return "saddle"
    if T ** 2 >= 4 * D:                            # real eigenvalues, same sign
        return ...
    if abs(T) < 1e-12:
        return "centre"
    return ...

for M in ([[-2, 0], [0, -1]], [[1, 0], [0, -1]], [[-0.5, 2], [-2, -0.5]], [[0, 1], [-1, 0]]):
    print(M, "->", classify(np.array(M, float)))

In [ ]:
import numpy as np
def test_cases():
    "classify names a stable node, saddle, stable spiral, unstable spiral and centre correctly"
    assert classify(np.array([[-2.0, 0], [0, -1]])) == "stable node"
    assert classify(np.array([[1.0, 0], [0, -1]])) == "saddle"
    assert classify(np.array([[-0.5, 2], [-2, -0.5]])) == "stable spiral"
    assert classify(np.array([[0.3, 1], [-1, 0.3]])) == "unstable spiral"
    assert classify(np.array([[0.0, 1], [-1, 0]])) == "centre"

def test_against_eig():
    "on 500 random matrices, classify agrees with the eigenvalues"
    r = np.random.default_rng(5)
    for _ in range(500):
        M = r.normal(size=(2, 2)); lam = np.linalg.eigvals(M); c = classify(M)
        if np.all(np.abs(lam.imag) < 1e-12) and lam.real.min() < 0 < lam.real.max(): assert c == "saddle"
        elif np.any(np.abs(lam.imag) > 1e-12): assert c.endswith("spiral") or c == "centre"

test_cases()
test_against_eig()
print("All checks passed")

### Exercise: Predicting the ringing

With tau_E = 10 ms, tau_I = 5 ms, wEE = 2, wEI = 3, wIE = 3 and wII = 1, build `A_ei`, predict the frequency `f_pred` in Hz from the imaginary part of its eigenvalues, then compute x_E(t) after a kick of x = (1, 0) using the eigen-decomposition, and measure `f_measured` from the spacing of its zero crossings over 0 to 60 ms.

In [ ]:
tE, tI, wEE, wEI, wIE, wII = 10.0, 5.0, 2.0, 3.0, 3.0, 1.0
A_ei = np.array([[(wEE - 1) / tE, -wEI / tE],
                 [wIE / tI, -(1 + wII) / tI]])
lam, V = np.linalg.eig(A_ei)
f_pred = ...                                      # Hz; time is in ms
ts = np.arange(0, 60, 0.01)
c = np.linalg.solve(V, np.array([1.0, 0.0]))
xE = np.real((V[0] * c) @ np.exp(np.outer(lam, ts)))
z = np.where(np.sign(xE[:-1]) != np.sign(xE[1:]))[0]
t_cross = ts[z] - xE[z] * (ts[z + 1] - ts[z]) / (xE[z + 1] - xE[z])     # linear interpolation
f_measured = ...                                  # successive crossings are half a period apart
print(f"eigenvalues {np.round(lam, 3)}; predicted {f_pred:.1f} Hz, measured {f_measured:.1f} Hz; decay time {-1 / lam.real[0]:.1f} ms")
plt.plot(ts, xE); plt.axhline(0, c="k", lw=0.8); plt.xlabel("ms"); plt.ylabel("x_E"); plt.show()

In [ ]:
import numpy as np
def test_kind():
    "the circuit is a stable spiral"
    assert classify(A_ei) == "stable spiral"

def test_freq():
    "the frequency measured from zero crossings is within 2% of the eigenvalue prediction"
    assert abs(f_measured - f_pred) / f_pred < 0.02

test_kind()
test_freq()
print("All checks passed")

### Exercise: Downhill all the way

Record the largest imaginary part of any eigenvalue over 1,000 random symmetric 5 × 5 matrices in `max_imag`. Then simulate x' = (W − I)x for a stable symmetric W (Euler, dt = 0.01, 2,000 steps) and store the energy at every step in `E_traj`. Repeat with an asymmetric W of the same size and store `E_asym`.

In [ ]:
r_e = np.random.default_rng(1984)
max_imag = 0.0
for _ in range(1000):
    M = r_e.normal(size=(5, 5)); S = ...           # a symmetric matrix built from M
    max_imag = max(max_imag, np.abs(np.linalg.eigvals(S).imag).max())

def energy_run(W, x0, steps=2000, dt=0.01):
    A = W - np.eye(len(W)); x = x0.copy(); Es = []
    for _ in range(steps):
        Es.append(...)                             # E = -1/2 x^T (W - I) x
        x = x + dt * A @ x
    return np.array(Es)

M = r_e.normal(size=(5, 5)); W_sym = 0.3 * (M + M.T) / 2
W_asym = np.zeros((5, 5)); W_asym[0, 1] = 5.0          # neuron 2 drives neuron 1, not the reverse
x0 = r_e.normal(size=5)
E_traj, E_asym = energy_run(W_sym, x0), energy_run(W_asym, x0)
print(f"largest imaginary part {max_imag:.1e};  symmetric: energy rises at {np.sum(np.diff(E_traj) > 1e-12)} steps, asymmetric: at {np.sum(np.diff(E_asym) > 1e-6)} steps")
plt.plot(E_traj, label="symmetric"); plt.plot(E_asym, label="asymmetric"); plt.xlabel("step"); plt.ylabel("E"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_real():
    "none of the 1,000 random symmetric matrices has a complex eigenvalue"
    assert max_imag < 1e-10

def test_downhill():
    "the energy never increases along the simulated trajectory"
    assert np.all(np.diff(E_traj) <= 1e-12)

def test_asym():
    "the asymmetric network's energy does increase at some point"
    assert np.any(np.diff(E_asym) > 1e-6)

test_real()
test_downhill()
test_asym()
print("All checks passed")